In [92]:
# =========================================================
# INSTALL LIBRARY
# =========================================================

!pip uninstall -y transformers peft accelerate -q

!pip install transformers==4.41.2 -q
!pip install accelerate==0.30.1 -q
!pip install peft==0.11.1 -q

!pip install sentencepiece datasets evaluate nltk -q

In [93]:
!pip install git+https://github.com/indobenchmark/indobenchmark-toolkit

  Cloning https://github.com/indobenchmark/indobenchmark-toolkit to /tmp/pip-req-build-fkrd9tzh
  Running command git clone --filter=blob:none --quiet https://github.com/indobenchmark/indobenchmark-toolkit /tmp/pip-req-build-fkrd9tzh
  Resolved https://github.com/indobenchmark/indobenchmark-toolkit to commit d519d9080c247764dcb3a2b45883ba1e90a8e6a9
  Preparing metadata (setup.py) ... done


In [94]:
# =========================================================
# IMPORT
# =========================================================

import pandas as pd
import torch
import shutil
import re

from sklearn.model_selection import train_test_split

from datasets import Dataset
from indobenchmark import IndoNLGTokenizer

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

In [95]:
# =========================================================
# HAPUS CHECKPOINT LAMA
# =========================================================

shutil.rmtree("./indobart-qg", ignore_errors=True)


In [96]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("H5_dataset_ML.csv")

In [97]:
# =========================================================
# BERSIHKAN DATA
# =========================================================

df["input"] = df["input"].astype(str).str.strip()
df["target"] = df["target"].astype(str).str.strip()

df = df[
    (df["input"] != "") &
    (df["target"] != "")
]

df = df.drop_duplicates()

df = df.reset_index(drop=True)

In [98]:
## =========================================================
# INFO
# =========================================================

print("=" * 50)
print("JUMLAH DATA")
print("=" * 50)

print(len(df))

print("\nCONTOH:")
print(df.head())

JUMLAH DATA
4674

CONTOH:
                                               input  \
0   generate siapa: Pagi itu Rina bangun lebih awal.   
1   generate kapan: Pagi itu Rina bangun lebih awal.   
2  generate siapa: Rina merapikan tempat tidur di...   
3  generate apa: Rina merapikan tempat tidur di k...   
4  generate dimana: Rina merapikan tempat tidur d...   

                                            target  
0           Siapa yang bangun pagi itu lebih awal?  
1                              Kapan Rina bangun ?  
2  Siapa yang merapikan tempat tidur di kamar nya?  
3              Apa yang Rina rapikan di kamar nya?  
4             Di mana Rina merapikan tempat tidur?  


In [99]:
# =========================================================
# SPLIT DATASET
# =========================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

In [100]:
# =========================================================
# SIMPAN TEST DATASET
# =========================================================

test_df.to_csv(
    "H6_test_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTest dataset berhasil disimpan")


Test dataset berhasil disimpan


In [10]:
# =========================================================
# CONVERT HUGGINGFACE DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [101]:
model_name = "indobenchmark/indobart"

tokenizer = IndoNLGTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

In [102]:
# =========================================================
# TOKENISASI
# =========================================================

max_input_length = 128
max_target_length = 64

def preprocess_function(examples):

    inputs = examples["input"]

    targets = examples["target"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # ubah padding menjadi -100
    labels["input_ids"] = [

        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]

        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [103]:
# =========================================================
# TOKENIZE DATASET
# =========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/3739 [00:00<?, ? examples/s]

Map:   0%|          | 0/467 [00:00<?, ? examples/s]

Map:   0%|          | 0/468 [00:00<?, ? examples/s]

In [104]:
# =========================================================
# CEK HASIL TOKENISASI
# =========================================================

print("\nHASIL TOKENISASI:")
print(tokenized_train[0])

print("\nLABEL SAMPLE:")
print(tokenized_train[0]["labels"][:20])



HASIL TOKENISASI:
{'input': 'generate apa: Vina menemukan empat sudut gambar.', 'target': 'Apa yang Vina temukan?', '__index_level_0__': 2715, 'input_ids': [8519, 1607, 597, 39967, 32412, 1647, 2380, 4494, 1534, 39954, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [597, 291, 

In [105]:
# =========================================================
# TRAINING ARGUMENT
# =========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./indobart-qg",

    evaluation_strategy="epoch",

    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    predict_with_generate=True,

    logging_steps=10,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [108]:
# =========================================================
# TRAINER
# =========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    # tokenizer=tokenizer
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [109]:
# =========================================================
# TRAIN MODEL
# =========================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.088500,0.363916
2,0.164300,0.326956
3,0.109400,0.320708


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generatio

TrainOutput(global_step=2805, training_loss=0.1268229765103675, metrics={'train_runtime': 561.031, 'train_samples_per_second': 19.994, 'train_steps_per_second': 5.0, 'total_flos': 854953195732992.0, 'train_loss': 0.1268229765103675, 'epoch': 3.0})

In [110]:
# =========================================================
# SAVE MODEL
# =========================================================

model.save_pretrained(
    "model_indobart_qg"
)

print("\nMODEL BERHASIL DISIMPAN")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}



MODEL BERHASIL DISIMPAN


In [154]:
# =========================================================
# TEST GENERATE
# =========================================================

text = "Pada suatu hari, Kancil berjalan santai di pinggir sungai."

input_text = (
    "generate apa: "
    + text
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

device = model.device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

In [155]:
# =========================================================
# GENERATE
# =========================================================

output_ids = model.generate(

    **inputs,

    max_new_tokens=20,

    num_beams=5,

    no_repeat_ngram_size=3,

    repetition_penalty=3.0,

    length_penalty=1.2,

    early_stopping=True
)

In [156]:
def clean_question(text):

    text = text.strip()

    # hapus tanda baca berulang
    text = re.sub(r'[?!.]{2,}', '?', text)

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text)


    # hapus koma di awal
    text = re.sub(r'^,\s*', '', text)

    # hapus koma sebelum tanda tanya
    text = re.sub(r',\s*\?', '?', text)

    # hapus koma berulang
    text = re.sub(r'\s*,\s*', ' ', text)

    # ambil hanya kalimat pertama
    text = text.split("?")[0] + "?"

    return text

In [157]:
# =========================================================
# DECODE
# =========================================================

hasil = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
    # clean_up_tokenization_spaces=True
)

hasil = clean_question(hasil)

In [158]:
# =========================================================
# OUTPUT
# =========================================================

print("\n" + "=" * 50)
print("HASIL GENERATE")
print("=" * 50)

print("INPUT:")
print(text)

print("\nOUTPUT:")
print(hasil)


HASIL GENERATE
INPUT:
Pada suatu hari, Kancil berjalan santai di pinggir sungai.

OUTPUT:
apa yang berjalan santai pada suatu hari di pinggir sungai?


In [130]:
# =========================================================
# LOAD MODEL TRAINED
# =========================================================

model_path = "model_indobart_qg"

# tokenizer tetap dari original model
tokenizer = IndoNLGTokenizer.from_pretrained(
    "indobenchmark/indobart"
)

# model dari hasil fine-tuning
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("\nModel loaded!")


Model loaded!


In [159]:
# =========================================================
# UPLOAD FILE TXT
# =========================================================

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving teks.txt to teks (2).txt


In [160]:
# =========================================================
# BACA FILE
# =========================================================

with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# normalize text
text = text.encode("utf-8", "ignore").decode("utf-8")

# hapus karakter aneh
text = re.sub(r'[^\w\s.,?!:;-]', ' ', text)

# rapikan spasi
text = re.sub(r'\s+', ' ', text)

text = text.strip()

print(repr(text[:500]))

'Pada suatu hari, Kancil berjalan santai di pinggir sungai. Perut Kancil terasa sangat lapar karena belum makan. Kancil melihat kebun mentimun yang segar di seberang sungai. Di dalam sungai itu, ada banyak buaya besar yang sedang tidur. Kancil mencari akal pintar agar bisa menyeberang dengan aman. Ia menyuruh para buaya berbaris lurus sampai ke tepi sungai seberang. Kancil berbohong bahwa ia disuruh raja hutan untuk membagikan daging. Para buaya percaya dan segera berbaris membentuk sebuah jembat'


In [161]:
# =========================================================
# SPLIT KALIMAT
# =========================================================

kalimat_list = re.split(
    r'(?<=[.!?])\s+',
    text
)

kalimat_list = [

    k.strip()

    for k in kalimat_list

    if len(k.strip()) > 3
]

print("\nJumlah kalimat:", len(kalimat_list))


Jumlah kalimat: 10


In [162]:
# =========================================================
# FUNCTION GENERATE
# =========================================================

def generate_question(text, tipe):

    input_text = f"generate {tipe}: {text}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(

        **inputs,

        max_new_tokens=20,

        num_beams=5,

        no_repeat_ngram_size=3,

        repetition_penalty=3.0,

        length_penalty=1.2,

        early_stopping=True
    )

    hasil = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        # clean_up_tokenization_spaces=True
    )
    hasil = hasil = clean_question(hasil)

    return hasil

In [151]:
# =========================================================
# DETEKSI TIPE PERTANYAAN
# =========================================================

def detect_relevant_types(text):

    text = text.lower()

    tipe = []

    tipe.append("Siapa")
    tipe.append("Apa")

    if " di " in f" {text} ":
        tipe.append("Di mana")

    if " ke " in f" {text} ":
        tipe.append("Ke mana")

    if " dari " in f" {text} ":
        tipe.append("Dari mana")

    waktu_keywords = [

        "pagi",
        "siang",
        "malam",
        "sore",
        "kemarin",
        "besok",
        "senin",
        "selasa",
        "rabu",
        "kamis",
        "jumat",
        "sabtu",
        "minggu",
        "pukul",
        "jam",
        "hari",
        "bulan",
        "menit"
    ]

    if any(k in text for k in waktu_keywords):
        tipe.append("Kapan")

    return tipe

In [163]:
# =========================================================
# GENERATE SEMUA
# =========================================================

hasil = []

for kalimat in kalimat_list:

    print("\n")
    print("=" * 60)

    print("KALIMAT:")
    print(kalimat)

    relevant_types = detect_relevant_types(
        kalimat
    )

    for tipe in relevant_types:

        try:

            pertanyaan = generate_question(
                kalimat,
                tipe
            )

            hasil.append({

                "kalimat": kalimat,

                "tipe": tipe,

                "pertanyaan": pertanyaan
            })

            print(f"\n[{tipe.upper()}]")
            print(pertanyaan)

        except Exception as e:

            print(f"ERROR {tipe}: {e}")



KALIMAT:
Pada suatu hari, Kancil berjalan santai di pinggir sungai.

[SIAPA]
siapa yang berjalan santai pada suatu hari di pinggir sungai?

[APA]
apa yang berjalan santai pada suatu hari di pinggir sungai?

[DI MANA]
di mana kancil berjalan ?

[KAPAN]
kapan kancil berjalan santai ?


KALIMAT:
Perut Kancil terasa sangat lapar karena belum makan.

[SIAPA]
siapa yang terasa sangat lapar karena belum makan?

[APA]
apa yang terasa sangat lapar karena belum makan?


KALIMAT:
Kancil melihat kebun mentimun yang segar di seberang sungai.

[SIAPA]
siapa yang melihat kebun mentimun yang segar di seberang sungai?

[APA]
apa yang melihat kebun mentimun yang segar di seberang sungai?

[DI MANA]
di mana kancil melihat kebun mentimun yang segar?


KALIMAT:
Di dalam sungai itu, ada banyak buaya besar yang sedang tidur.

[SIAPA]
siapa yang sedang tidur di dalam sungai itu ada banyak buaya besar yang sedang mandi?

[APA]
apa yang sedang tidur di dalam sungai itu sangat?

[DI MANA]
di mana ada buaya bes